# DocsGPT: Intelligent Documentation Assistant

This notebook builds a complete AI-powered documentation assistant step by step.
Each part focuses on one piece of the system and shows how the code works.

| Part | What it does |
|------|--------------|
| 1 | Load text from three real datasets |
| 2 | Summarise each document using DocsGPT |
| 3 | Generate FAQs from each document |
| 4 | Score the outputs with ROUGE and BLEU |
| 5 | Produce summaries and FAQs in other languages |
| 6 | Show all results in a summary table |
| 7 | Launch an interactive Gradio UI |

**Setup:**
```bash
./docker_build.sh
./docker_jupyter.sh
cp .env.example .env
# Open .env and set DOCSGPT_API_KEY=your-agent-key
```


In [2]:
%load_ext autoreload
%autoreload 2

import logging
import os
import pandas as pd

import docsgpt_utils as tdgputi

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
_LOG = logging.getLogger(__name__)

BASE_URL = tdgputi.get_base_url()
API_KEY  = tdgputi.get_api_key()

print(f"Base URL : {BASE_URL}")
print(f"API key  : {API_KEY[:10]}***  (truncated for safety)")
print("Environment ready ✓")

Base URL : https://gptcloud.arc53.com
API key  : 5ffa3a62-f***  (truncated for safety)
Environment ready ✓


---

## Part 1: Data Collection

We load text from three sources. Each one is handled by a function in
`docsgpt_utils.py` that fetches the data and returns plain text.

- **`fetch_awesome_ml_readme()`** — downloads the raw markdown file from GitHub
  using `requests.get()`. `parse_awesome_ml_sections()` then splits it into a
  dict of `{section_title: body_text}` by scanning for `## ` heading lines.
  `clean_markdown()` strips all markdown syntax (links, bullets, code blocks)
  so we're left with plain prose the LLM can read cleanly.

- **`load_stackoverflow_sample()`** — connects to HuggingFace Hub and streams
  rows one at a time using `datasets.load_dataset(..., streaming=True)`. Only
  the first `n_rows` rows are read — the full dataset is never downloaded.
  `so_rows_to_text()` combines the title, body, and answer of each row into
  one readable block of text.

- **`load_pile_sample()`** — streams The Pile dataset the same way, collecting
  rows until we have enough characters.

All three functions fall back to built-in sample data if the network is
unavailable, so the notebook always keeps running.


In [2]:
# ── 1a. Awesome Machine Learning (GitHub README) ───────────────────────────
# fetch_awesome_ml_readme() downloads the raw markdown from GitHub.
# parse_awesome_ml_sections() splits it into a dict: {section_title: body}
# clean_markdown() strips all markdown syntax → plain prose for the LLM.

print("=" * 60)
print("1a. Fetching Awesome Machine Learning README from GitHub...")

raw_md   = tdgputi.fetch_awesome_ml_readme()
sections = tdgputi.parse_awesome_ml_sections(raw_md)

print(f"    Total chars   : {len(raw_md):,}")
print(f"    Sections found: {len(sections)}")
print(f"    Section names : {list(sections.keys())[:6]} ...")

# Pick the largest content-rich section to give the LLM something meaty
content_sections = {k: v for k, v in sections.items() if len(v) > 500}
aml_title = sorted(content_sections, key=lambda k: len(content_sections[k]), reverse=True)[0]
aml_text  = tdgputi.clean_markdown(content_sections[aml_title])

print(f"    Selected      : '{aml_title}'")
print(f"    Clean text    : {len(aml_text):,} chars")
print(f"    Preview       : {aml_text[:200]}...")

2026-05-06 13:20:16,138 [INFO] docsgpt_utils: fetching Awesome ML README from https://raw.githubusercontent.com/josephmisiti/awesome-machine-learning/master/README.md
2026-05-06 13:20:16,304 [INFO] docsgpt_utils: fetched 212555 chars
2026-05-06 13:20:16,306 [INFO] docsgpt_utils: parsed 41 sections from Awesome ML README


1a. Fetching Awesome Machine Learning README from GitHub...
    Total chars   : 212,555
    Sections found: 41
    Section names : ['Preamble', 'IMPORTANT NOTE ON PRs:', 'Star History', 'Table of Contents', 'APL', 'C'] ...
    Selected      : 'Python'
    Clean text    : 17,302 chars
    Preview       : Computer Vision
LightlyTrain - Pretrain computer vision models on unlabeled data for industrial applications
Scikit-Image - A collection of algorithms for image processing in Python.
Scikit-Opt - Swar...


In [3]:
# ── 1b. Stack Overflow Questions (HuggingFace Hub, streaming) ──────────────
# load_stackoverflow_sample() uses the HuggingFace `datasets` library with
# streaming=True so we never download the full dataset — just the first N rows.
#
# so_rows_to_text() combines title + body + answer into one readable document
# per question, then joins them all with --- separators.

print("1b. Loading Stack Overflow questions from HuggingFace Hub...")
print("    (uses streaming — no full download needed)")

so_rows = tdgputi.load_stackoverflow_sample(n_rows=10)
so_text = tdgputi.so_rows_to_text(so_rows)

print(f"\n    Rows loaded   : {len(so_rows)}")
print(f"    Combined chars: {len(so_text):,}")
print(f"    First question: {so_rows[0]['title']}")
print(f"    First answer  : {so_rows[0]['answer'][:100]}...")

1b. Loading Stack Overflow questions from HuggingFace Hub...
    (uses streaming — no full download needed)


/Users/kshitid/Kshiti/UMCP/DATA605/docsgpt/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-06 13:20:24,214 [INFO] docsgpt_utils: loading SO dataset (streaming, n=10)
2026-05-06 13:20:29,282 [INFO] docsgpt_utils: loaded 10 SO rows



    Rows loaded   : 10
    Combined chars: 17,201
    First question: Parsing json directly using input stream
    First answer  : ...


In [4]:
# ── 1c. The Pile — uncopyrighted subset (HuggingFace Hub, streaming) ───────
# The Pile is a large-scale dataset of diverse text. We use the uncopyrighted
# subset for safety. Again we stream just enough characters rather than
# downloading the whole thing.

print("1c. Streaming a sample from The Pile (uncopyrighted subset)...")

pile_text = tdgputi.load_pile_sample(n_chars=8000)

print(f"    Collected : {len(pile_text):,} chars")
print(f"    Preview   : {pile_text[:200]}...")

# ── Package everything into a named dict for the rest of the notebook ───────
SOURCE_TEXTS = {
    "Awesome ML":    aml_text,
    "Stack Overflow": so_text,
    "The Pile":       pile_text,
}

print("\n📦 All datasets ready:")
for label, text in SOURCE_TEXTS.items():
    print(f"   • {label}: {len(text):,} chars")

2026-05-06 13:20:44,442 [INFO] docsgpt_utils: streaming Pile dataset (target 8000 chars)


1c. Streaming a sample from The Pile (uncopyrighted subset)...


2026-05-06 13:20:45,523 [WARNING] docsgpt_utils: Pile dataset unavailable: Compression type zstd not supported — using fallback


    Collected : 829 chars
    Preview   : Natural language processing (NLP) is a subfield of artificial intelligence that focuses on enabling computers to understand, interpret, and generate human language. Modern NLP relies on large transfor...

📦 All datasets ready:
   • Awesome ML: 17,302 chars
   • Stack Overflow: 17,201 chars
   • The Pile: 829 chars


---

## Part 2: Text Summarisation

`summarize_document()` takes a block of text and sends it to DocsGPT to
get a summary. Here is what happens inside the function:

1. `truncate_text()` cuts the text to 4000 characters so it fits in the prompt
2. The text is placed inside a prompt string that instructs DocsGPT to
   summarise it in a certain number of words
3. That prompt is sent to `POST /api/answer` via `query_docsgpt()`
4. The `"answer"` field from the JSON response is returned as the summary

The loop below runs this for each of the three datasets and stores the
results in a `summaries` dict keyed by dataset name.


In [5]:
# ── Summarise each dataset source ──────────────────────────────────────────
# summarize_document() builds the prompt, calls POST /api/answer, and returns
# the 'answer' field from the JSON response.

summaries: dict = {}

for label, text in SOURCE_TEXTS.items():
    print(f"\n{'='*60}")
    print(f"📄 Summarising: {label}  ({len(text):,} chars → truncated to 4000)")

    truncated = tdgputi.truncate_text(text, max_chars=4000)

    summary = tdgputi.summarize_document(
        truncated,
        API_KEY,
        BASE_URL,
        max_words=200,
        source_label=label,
    )
    summaries[label] = summary
    word_count = len(summary.split())
    print(f"\n✅ Summary ({word_count} words):")
    print(summary)

print(f"\n\n🎉 All {len(summaries)} summaries generated!")

2026-05-06 13:20:54,418 [INFO] docsgpt_utils: summarize_document: 'Awesome ML' (4012 chars)
2026-05-06 13:20:54,418 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 2



📄 Summarising: Awesome ML  (17,302 chars → truncated to 4000)


2026-05-06 13:20:56,187 [INFO] docsgpt_utils: answer received (1473 chars)
2026-05-06 13:20:56,188 [INFO] docsgpt_utils: summarize_document: 'Stack Overflow' (4012 chars)
2026-05-06 13:20:56,189 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 2



✅ Summary (173 words):
This document outlines a diverse ecosystem of Python-based libraries and frameworks dedicated to **Computer Vision (CV)**, ranging from foundational image processing to advanced deep learning.

Key categories include:
*   **General Frameworks & Image Processing:** Tools like **Scikit-Image**, **SimpleCV**, and **imutils** provide essential algorithms and convenience functions. **Detecto** and **PyTorchCV** offer streamlined workflows for model training.
*   **Face Recognition & Analysis:** High-performance libraries such as **deepface**, **face_recognition**, and **retinaface** enable facial detection, landmarking, and attribute analysis (age, gender, emotion).
*   **Object Detection & Segmentation:** FAIR’s **Detectron2** and **albumentations** serve as industry standards for detection, segmentation, and robust data augmentation.
*   **Generative AI & Style Transfer:** Resources like **TF-GAN**, **neural-style-pt**, and **joliGEN** support GANs, diffusion model

2026-05-06 13:20:58,556 [INFO] docsgpt_utils: answer received (1237 chars)
2026-05-06 13:20:58,557 [INFO] docsgpt_utils: summarize_document: 'The Pile' (829 chars)
2026-05-06 13:20:58,557 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 2



✅ Summary (178 words):
The user is seeking a memory-efficient method to parse JSON directly from an `InputStream` for Android applications targeting versions as low as 2.0. They aim to avoid loading the entire response into memory as a string, which is common with standard JSON objects.

While the native `android.util.JsonReader` is limited to API 11+, the user attempts to use the **GSON library's `JsonReader`** as a compatible alternative. However, their implementation fails with a `java.io.EOFException`. 

**Key Technical Issue:**
The provided code demonstrates that the `InputStream` is being consumed twice. First, the user reads the entire stream into a `StringBuilder` via a `BufferedReader` (intended for a `TextView`). When they subsequently pass the same stream to GSON’s `JsonReader`, the stream has already reached the end, causing the parser to fail immediately.

**Summary of Requirements:**
*   **Target:** Android API 2.0+.
*   **Goal:** Streaming JSON parsing to save memory.
*

2026-05-06 13:21:00,999 [INFO] docsgpt_utils: answer received (1007 chars)



✅ Summary (134 words):
Natural Language Processing (NLP) is a branch of artificial intelligence focused on the interaction between computers and human language. Contemporary NLP is driven by large **transformer-based models**—such as BERT, GPT, and T5—which are pretrained on massive datasets and fine-tuned for specialized tasks.

A pivotal development in the field was the **attention mechanism** (Vaswani et al., 2017), which allows models to weigh the importance of different words within a sequence. This, combined with **transfer learning**, has significantly reduced the amount of labeled data required for high-performance results.

Key NLP tasks include:
*   **Text classification** and **sentiment analysis**.
*   **Named entity recognition (NER)** and **question answering**.
*   **Machine translation** and **text summarization**.

Today, NLP powers essential technologies such as search engines, virtual assistants, chatbots, and automated content moderation, making it a cornerstone of

---

## Part 3: FAQ Generation

`generate_faqs()` works the same way as summarisation — it embeds the
document text in a prompt and sends it to `POST /api/answer`. The prompt
instructs DocsGPT to format the output as:

```
Q: <question>
A: <answer>
```

Once the response comes back, `parse_faqs()` processes the raw text:
1. Splits the text on `Q:` markers using `re.split()`
2. For each block, extracts the question with a `re.search()` for `Q: ...`
   and the answer with a search for `A: ...`
3. Returns a list of `{"question": ..., "answer": ...}` dicts

`print_faqs()` then formats and prints each pair.


In [6]:
# ── Generate FAQs for each dataset source ─────────────────────────────────
# generate_faqs() sends a structured prompt to /api/answer requesting
# exactly n_questions FAQs in Q:/A: format, then calls parse_faqs() on the result.

all_faqs: dict = {}

for label, text in SOURCE_TEXTS.items():
    print(f"\n{'='*60}")
    print(f"❓ Generating FAQs: {label}")

    truncated = tdgputi.truncate_text(text, max_chars=4000)

    faqs = tdgputi.generate_faqs(
        truncated,
        API_KEY,
        BASE_URL,
        n_questions=4,
        source_label=label,
    )
    all_faqs[label] = faqs

    print(f"\n✅ Generated {len(faqs)} FAQs:")
    tdgputi.print_faqs(faqs)

print(f"\n\n🎉 FAQ generation complete for all {len(all_faqs)} sources!")

2026-05-06 13:21:15,974 [INFO] docsgpt_utils: generate_faqs: 4 Qs for 'Awesome ML'
2026-05-06 13:21:15,974 [INFO] docsgpt_utils: query_docsgpt | Based on the document below, generate exactly 4 frequently asked questions (FAQs) with detailed, hel



❓ Generating FAQs: Awesome ML


2026-05-06 13:21:17,892 [INFO] docsgpt_utils: answer received (2298 chars)
2026-05-06 13:21:17,894 [INFO] docsgpt_utils: generate_faqs: 4 Qs for 'Stack Overflow'
2026-05-06 13:21:17,895 [INFO] docsgpt_utils: query_docsgpt | Based on the document below, generate exactly 4 frequently asked questions (FAQs) with detailed, hel



✅ Generated 4 FAQs:

Q1: What is the difference between Detectron and Detectron2 according to the document?
A1: Detectron is Facebook AI Research's (FAIR) original software system for object detection, which implements algorithms like Mask R-CNN and is powered by the Caffe2 framework; however, it is now labeled as deprecated. Detectron2 is the next-generation research platform from FAIR, serving as a ground-up rewrite of the original version. Unlike its predecessor, Detectron2 is powered by the PyTorch deep learning framework and supports advanced object detection and segmentation tasks.

Q2: Which libraries are recommended for performing face recognition and facial attribute analysis?
A2: There are several libraries listed for these tasks. The "face_recognition" library is designed for recognizing and manipulating faces from Python or the command line. For more comprehensive analysis, "deepface" is a lightweight framework that handles recognition and facial attribute analysis—such as

2026-05-06 13:21:20,391 [INFO] docsgpt_utils: answer received (2050 chars)
2026-05-06 13:21:20,392 [INFO] docsgpt_utils: generate_faqs: 4 Qs for 'The Pile'
2026-05-06 13:21:20,392 [INFO] docsgpt_utils: query_docsgpt | Based on the document below, generate exactly 4 frequently asked questions (FAQs) with detailed, hel



✅ Generated 4 FAQs:

Q1: Why should I parse JSON directly from an InputStream instead of converting it to a String first?
A1: Parsing JSON directly from an InputStream is much more memory-efficient, especially for large datasets. By using a streaming parser like JsonReader, you process the data token by token as it arrives rather than loading the entire JSON payload into a String in your device's RAM. This prevents OutOfMemoryErrors that often occur when handling large API responses on mobile devices.

Q2: How can I implement JSON streaming on Android versions older than API level 11?
A2: While the native `android.util.JsonReader` requires API level 11 or higher, you can support older versions (Android 2.0 and up) by using the Google GSON library. By including the GSON dependency, you can use `com.google.gson.stream.JsonReader`, which provides the same streaming functionality and API as the native version but remains compatible with legacy Android releases.

Q3: Why does my code throw

2026-05-06 13:21:22,402 [INFO] docsgpt_utils: answer received (1464 chars)



✅ Generated 4 FAQs:

Q1: What is natural language processing (NLP) and what is its primary goal?
A1: Natural language processing (NLP) is a specialized subfield of artificial intelligence. Its primary goal is to enable computers to understand, interpret, and generate human language, allowing for more seamless interaction between humans and machines.

Q2: Which models are commonly used in modern NLP, and how are they developed?
A2: Modern NLP relies on large transformer-based models such as BERT, GPT, and T5. These models are developed through a two-step process: first, they are pretrained on massive corpora of text data, and then they are fine-tuned to perform specific tasks such as text classification or sentiment analysis.

Q3: What is the significance of the attention mechanism in NLP?
A3: The attention mechanism, which was introduced in the 2017 paper 'Attention is All You Need' by Vaswani et al., is a breakthrough that allows models to weigh the importance of different words acro

---

## Part 4: Evaluation — ROUGE and BLEU

We score each generated summary and FAQ answer against the original source
text to measure how well the content was captured.

**`rouge_scores(hypothesis, reference)`** uses the `rouge_score` library to
compute three variants of ROUGE:
- **ROUGE-1**: counts how many individual words overlap
- **ROUGE-2**: counts how many two-word pairs overlap
- **ROUGE-L**: finds the longest matching sequence of words in order

All three are F1 scores (0 to 1). A higher number means more overlap
with the reference text.

**`bleu_score(hypothesis, reference)`** uses NLTK to compute BLEU, which
measures how precisely the generated text matches n-grams in the reference.
It also penalises outputs that are too short.

**`evaluate_all()`** runs both metrics for every dataset. It uses the first
500 characters of each source as the reference, scores the summary, then
scores the first FAQ answer, and collects everything into a results dict.


In [7]:
# ── Evaluate all summaries and FAQs ────────────────────────────────────────
# evaluate_all() runs evaluate_output() for each document label.
# evaluate_output() calls rouge_scores() and bleu_score() internally.
# The reference for each document is its first 500 chars.

print("📊 Running ROUGE + BLEU evaluation...\n")

eval_results = tdgputi.evaluate_all(summaries, all_faqs, SOURCE_TEXTS)

for label, scores in eval_results.items():
    print(f"\n[{label}]")
    print(f"  Summary scores:")
    print(f"    ROUGE-1 : {scores.get('rouge1', 0):.4f}")
    print(f"    ROUGE-2 : {scores.get('rouge2', 0):.4f}")
    print(f"    ROUGE-L : {scores.get('rougeL', 0):.4f}")
    print(f"    BLEU    : {scores.get('bleu',   0):.4f}")
    if 'faq_rouge1' in scores:
        print(f"  FAQ answer scores (first FAQ):")
        print(f"    ROUGE-1 : {scores.get('faq_rouge1', 0):.4f}")
        print(f"    BLEU    : {scores.get('faq_bleu',   0):.4f}")

2026-05-06 13:21:36,853 [INFO] absl: Using default tokenizer.
2026-05-06 13:21:37,011 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.214, 'rouge2': 0.0415, 'rougeL': 0.0823, 'bleu': 0.0042}
2026-05-06 13:21:37,012 [INFO] absl: Using default tokenizer.
2026-05-06 13:21:37,013 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.1, 'rouge2': 0.0, 'rougeL': 0.0571, 'bleu': 0.0039}
2026-05-06 13:21:37,013 [INFO] absl: Using default tokenizer.
2026-05-06 13:21:37,016 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.2657, 'rouge2': 0.0493, 'rougeL': 0.1189, 'bleu': 0.0053}
2026-05-06 13:21:37,016 [INFO] absl: Using default tokenizer.
2026-05-06 13:21:37,017 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.2561, 'rouge2': 0.0247, 'rougeL': 0.1341, 'bleu': 0.0112}
2026-05-06 13:21:37,017 [INFO] absl: Using default tokenizer.
2026-05-06 13:21:37,019 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.5507, 'rouge2': 0.361, 'rougeL': 0.4638, 'bleu': 0.1065}
2026-05-06 13:21:37,019 [INF

📊 Running ROUGE + BLEU evaluation...


[Awesome ML]
  Summary scores:
    ROUGE-1 : 0.2140
    ROUGE-2 : 0.0415
    ROUGE-L : 0.0823
    BLEU    : 0.0042
  FAQ answer scores (first FAQ):
    ROUGE-1 : 0.1000
    BLEU    : 0.0039

[Stack Overflow]
  Summary scores:
    ROUGE-1 : 0.2657
    ROUGE-2 : 0.0493
    ROUGE-L : 0.1189
    BLEU    : 0.0053
  FAQ answer scores (first FAQ):
    ROUGE-1 : 0.2561
    BLEU    : 0.0112

[The Pile]
  Summary scores:
    ROUGE-1 : 0.5507
    ROUGE-2 : 0.3610
    ROUGE-L : 0.4638
    BLEU    : 0.1065
  FAQ answer scores (first FAQ):
    ROUGE-1 : 0.4190
    BLEU    : 0.1244


---

## Part 5: Multi-Language Support

`summarize_multilang()` and `generate_faqs_multilang()` produce output in
any of the 9 supported languages by running translation before and after
the DocsGPT call.

**`translate_text(text, source, target)`** uses the `deep-translator`
library which calls the Google Translate API. If the text is longer than
4500 characters (Google's limit per request), the function splits it on
sentence boundaries using `re.split()`, translates each batch separately,
then joins the results back together.

The full pipeline for `summarize_multilang()` is:
1. If the source language is not English, translate the input text to English
2. Call `summarize_document()` to get an English summary from DocsGPT
3. If the output language is not English, translate the summary to the target
4. Return both the English and translated summaries in a dict

`generate_faqs_multilang()` follows the same pipeline but translates each
FAQ question and answer individually.


In [12]:
# ── Multi-language summarisation demo ──────────────────────────────────────
print("🌍 Supported languages:")
for code, name in tdgputi.list_supported_languages().items():
    print(f"   {code}: {name}")

# Use the Stack Overflow text for this demo
demo_text   = tdgputi.truncate_text(SOURCE_TEXTS["Stack Overflow"], max_chars=2000)
target_langs = ["es", "fr", "de"]

multilang_results = {}

for lang_code in target_langs:
    lang_name = tdgputi.SUPPORTED_LANGUAGES[lang_code]
    print(f"\n{'='*55}")
    print(f"🌐 Target: {lang_name} ({lang_code})")

    result = tdgputi.summarize_multilang(
        demo_text,
        API_KEY,
        source_lang="en",
        output_lang=lang_code,
        base_url=BASE_URL,
        max_words=100,
        source_label=f"so_{lang_code}",
    )
    multilang_results[lang_code] = result

    print(f"\n[English summary]")
    print(result['english_summary'])
    print(f"\n[{lang_name} translation]")
    print(result['translated_summary'])

print("\n\n✅ Multi-language summaries complete!")

2026-05-06 13:24:11,019 [INFO] docsgpt_utils: summarize_document: 'so_es' (2011 chars)
2026-05-06 13:24:11,019 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 1


🌍 Supported languages:
   en: English
   es: Spanish
   fr: French
   de: German
   zh: Chinese
   pt: Portuguese
   it: Italian
   ja: Japanese
   ar: Arabic

🌐 Target: Spanish (es)


2026-05-06 13:24:12,589 [INFO] docsgpt_utils: answer received (565 chars)
2026-05-06 13:24:12,650 [INFO] docsgpt_utils: translating 565 chars: en -> es
2026-05-06 13:24:13,717 [INFO] docsgpt_utils: summarize_document: 'so_fr' (2011 chars)
2026-05-06 13:24:13,718 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 1



[English summary]
The user seeks a memory-efficient method to parse JSON directly from an **InputStream** on Android, specifically targeting compatibility with version 2.0 (API level 5) and above. While trying to avoid loading large strings into memory, they found that the native `JsonReader` requires API level 11. Their current implementation incorrectly consumes the stream into a `StringBuilder` before parsing, defeating the purpose of streaming. They are looking for a solution using **GSON** or a similar library that supports stream-based parsing for older Android versions.

[Spanish translation]
El usuario busca un método eficiente en memoria para analizar JSON directamente desde un **InputStream** en Android, específicamente dirigido a la compatibilidad con la versión 2.0 (API nivel 5) y superiores. Mientras intentaban evitar cargar cadenas grandes en la memoria, descubrieron que el `JsonReader` nativo requiere el nivel de API 11. Su implementación actual consume incorrectamente 

2026-05-06 13:24:15,147 [INFO] docsgpt_utils: answer received (565 chars)
2026-05-06 13:24:15,148 [INFO] docsgpt_utils: translating 565 chars: en -> fr
2026-05-06 13:24:15,293 [INFO] docsgpt_utils: summarize_document: 'so_de' (2011 chars)
2026-05-06 13:24:15,294 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 1



[English summary]
The user seeks a memory-efficient method to parse JSON directly from an **InputStream** on Android, specifically targeting compatibility with version 2.0 (API level 5) and above. While trying to avoid loading large strings into memory, they found that the native `JsonReader` requires API level 11. Their current implementation incorrectly consumes the stream into a `StringBuilder` before parsing, defeating the purpose of streaming. They are looking for a solution using **GSON** or a similar library that supports stream-based parsing for older Android versions.

[French translation]
L'utilisateur recherche une méthode économe en mémoire pour analyser JSON directement à partir d'un **InputStream** sur Android, en ciblant spécifiquement la compatibilité avec la version 2.0 (API niveau 5) et supérieure. Tout en essayant d'éviter de charger de grandes chaînes en mémoire, ils ont constaté que le « JsonReader » natif nécessite le niveau d'API 11. Leur implémentation actuelle

2026-05-06 13:24:17,197 [INFO] docsgpt_utils: answer received (565 chars)
2026-05-06 13:24:17,198 [INFO] docsgpt_utils: translating 565 chars: en -> de



[English summary]
The user seeks a memory-efficient method to parse JSON directly from an **InputStream** on Android, specifically targeting compatibility with version 2.0 (API level 5) and above. While trying to avoid loading large strings into memory, they found that the native `JsonReader` requires API level 11. Their current implementation incorrectly consumes the stream into a `StringBuilder` before parsing, defeating the purpose of streaming. They are looking for a solution using **GSON** or a similar library that supports stream-based parsing for older Android versions.

[German translation]
Der Benutzer sucht nach einer speichereffizienten Methode, um JSON direkt aus einem **InputStream** auf Android zu analysieren, und strebt dabei insbesondere die Kompatibilität mit Version 2.0 (API-Level 5) und höher an. Beim Versuch, das Laden großer Strings in den Speicher zu vermeiden, stellten sie fest, dass der native „JsonReader“ API-Level 11 erfordert. Ihre aktuelle Implementierung v

---

## Part 6: Results Dashboard

This cell collects all the outputs generated so far — summaries, FAQ counts,
and evaluation scores — and arranges them into a pandas DataFrame.

Each row represents one dataset. The columns show the number of FAQs
generated, a preview of the summary, and the four metric scores.
`pd.set_option()` controls how wide the summary preview column is and how
many decimal places the scores display.


In [13]:
# ── Results dashboard ──────────────────────────────────────────────────────
rows = []
for label in SOURCE_TEXTS:
    scores  = eval_results.get(label, {})
    n_faqs  = len(all_faqs.get(label, []))
    summary = summaries.get(label, "")
    rows.append({
        "Document":  label,
        "# FAQs":    n_faqs,
        "Summary preview": (summary[:80] + "...") if len(summary) > 80 else summary,
        "ROUGE-1":   scores.get("rouge1", 0.0),
        "ROUGE-2":   scores.get("rouge2", 0.0),
        "ROUGE-L":   scores.get("rougeL", 0.0),
        "BLEU":      scores.get("bleu",   0.0),
    })

dashboard = pd.DataFrame(rows).set_index("Document")
pd.set_option("display.max_colwidth", 85)
pd.set_option("display.float_format", "{:.4f}".format)

print("=" * 70)
print("       DocsGPT Documentation Assistant — Results Dashboard")
print("=" * 70)
print(dashboard.to_string())
print("=" * 70)

best_rouge  = dashboard["ROUGE-1"].idxmax()
best_bleu   = dashboard["BLEU"].idxmax()
print(f"\n🏆 Best ROUGE-1 : {best_rouge}  ({dashboard.loc[best_rouge, 'ROUGE-1']:.4f})")
print(f"🏆 Best BLEU    : {best_bleu}  ({dashboard.loc[best_bleu, 'BLEU']:.4f})")

       DocsGPT Documentation Assistant — Results Dashboard
                # FAQs                                                                      Summary preview  ROUGE-1  ROUGE-2  ROUGE-L   BLEU
Document                                                                                                                                     
Awesome ML           4  This document outlines a diverse ecosystem of Python-based libraries and framewo...   0.2140   0.0415   0.0823 0.0042
Stack Overflow       4  The user is seeking a memory-efficient method to parse JSON directly from an `In...   0.2657   0.0493   0.1189 0.0053
The Pile             4  Natural Language Processing (NLP) is a branch of artificial intelligence focused...   0.5507   0.3610   0.4638 0.1065

🏆 Best ROUGE-1 : The Pile  (0.5507)
🏆 Best BLEU    : The Pile  (0.1065)


---

## Part 7: Gradio User Interface

This cell builds an interactive web UI using Gradio. When a user clicks
the Generate button, the `run_docsgpt_ui()` function runs:

1. Truncates the input text to 3500 characters
2. Calls `summarize_multilang()` to get a summary in the selected language
3. Calls `generate_faqs_multilang()` to get FAQs in the selected language
4. Calls `evaluate_output()` to compute ROUGE and BLEU scores for the summary
5. Returns all three results to the UI components

`gr.Blocks()` defines the layout — a text input, a language dropdown, a
slider for FAQ count, and three output areas. `submit_btn.click()` wires
the button to the function, specifying which inputs to read and which
outputs to update.

Run the cell, then open **http://127.0.0.1:7860** in your browser.


In [ ]:
# ── Gradio UI ──────────────────────────────────────────────────────────────
import gradio as gr


def run_docsgpt_ui(document_text: str, output_language: str, n_faqs: int) -> tuple:
    """Gradio handler: summarise + generate FAQs + evaluate + translate."""
    if not document_text.strip():
        return "⚠️ Please paste a document first.", "", ""

    lang_map      = {v: k for k, v in tdgputi.list_supported_languages().items()}
    out_lang_code = lang_map.get(output_language, "en")

    try:
        truncated = tdgputi.truncate_text(document_text, max_chars=3500)

        # ── Summarisation ──────────────────────────────────────────────────
        result          = tdgputi.summarize_multilang(
            truncated, API_KEY, source_lang="en",
            output_lang=out_lang_code, base_url=BASE_URL, source_label="ui",
        )
        english_summary    = result["english_summary"]
        translated_summary = result["translated_summary"]

        summary_md = (
            f"### Summary\n{english_summary}" if out_lang_code == "en"
            else f"### English Summary\n{english_summary}\n\n### {output_language} Summary\n{translated_summary}"
        )

        # ── FAQ Generation ─────────────────────────────────────────────────
        faq_result = tdgputi.generate_faqs_multilang(
            truncated, API_KEY, source_lang="en",
            output_lang=out_lang_code, n_questions=int(n_faqs),
            base_url=BASE_URL, source_label="ui_faq",
        )
        faq_lines = [
            f"**Q{i}: {faq['question']}**\nA: {faq['answer']}"
            for i, faq in enumerate(faq_result["translated_faqs"], 1)
        ]
        faqs_md = "\n\n".join(faq_lines) or "No FAQs generated."

        # ── Evaluation ─────────────────────────────────────────────────────
        scores      = tdgputi.evaluate_output(english_summary, document_text[:500])
        scores_text = "\n".join(f"{k.upper()}: {v:.4f}" for k, v in scores.items())

        return summary_md, faqs_md, scores_text

    except Exception as exc:
        return f"❌ Error: {exc}", "", ""


# ── Build the interface ────────────────────────────────────────────────────
lang_choices = list(tdgputi.list_supported_languages().values())

EXAMPLE_TEXT = (
    "Python is a high-level, interpreted programming language known for its "
    "clear syntax and readability. It supports multiple programming paradigms "
    "including procedural, object-oriented, and functional programming. Python "
    "is widely used in data science, machine learning, web development, and "
    "automation. It was created by Guido van Rossum and first released in 1991. "
    "Python has a large standard library and a vibrant open-source ecosystem. "
    "Its package manager pip provides access to hundreds of thousands of packages."
)

with gr.Blocks(title="DocsGPT Documentation Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 📄 DocsGPT Documentation Assistant\n"
        "Paste any technical document to get an AI-powered **summary** and **FAQs** "
        "in your chosen language — powered by **DocsGPT Cloud** (`POST /api/answer`)."
    )

    with gr.Row():
        with gr.Column(scale=3):
            doc_input = gr.Textbox(
                label="📋 Document Text",
                placeholder="Paste your technical document here...",
                lines=12, value=EXAMPLE_TEXT,
            )
        with gr.Column(scale=1):
            lang_dropdown = gr.Dropdown(
                choices=lang_choices, value="English", label="🌍 Output Language",
            )
            n_faqs_slider = gr.Slider(
                minimum=1, maximum=8, value=4, step=1, label="❓ Number of FAQs",
            )
            submit_btn = gr.Button("🚀 Generate Summary + FAQs", variant="primary", size="lg")

    with gr.Row():
        summary_out = gr.Markdown(label="Summary")

    with gr.Row():
        faqs_out = gr.Markdown(label="FAQs")

    scores_out = gr.Textbox(
        label="📊 Evaluation Scores (ROUGE + BLEU)", lines=5, interactive=False,
    )

    submit_btn.click(
        fn=run_docsgpt_ui,
        inputs=[doc_input, lang_dropdown, n_faqs_slider],
        outputs=[summary_out, faqs_out, scores_out],
    )

print("Launching Gradio app at http://127.0.0.1:7860 ...")
demo.launch(share=True)
# Set share=True to get a public URL for demos

/var/folders/lj/89tkjhqd6034t1cfsz5drsy80000gn/T/ipykernel_17203/3298890697.py:64: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="DocsGPT Documentation Assistant", theme=gr.themes.Soft()) as demo:
2026-05-06 19:27:42,318 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-05-06 19:27:42,337 [INFO] httpx: HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-05-06 19:27:42,344 [INFO] httpx: HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


Launching Gradio app at http://127.0.0.1:7860 ...
* Running on local URL:  http://127.0.0.1:7860


2026-05-06 19:27:42,662 [INFO] httpx: HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-05-06 19:27:42,716 [INFO] httpx: HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-05-06 19:27:42,990 [INFO] httpx: HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_darwin_arm64 "HTTP/1.1 200 OK"


* Running on public URL: https://a6a2d2994f3cbb2171.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-05-06 19:27:44,691 [INFO] httpx: HTTP Request: HEAD https://a6a2d2994f3cbb2171.gradio.live "HTTP/1.1 200 OK"


2026-05-06 19:27:44,755 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-05-06 21:57:57,878 [INFO] docsgpt_utils: summarize_document: 'ui' (514 chars)
2026-05-06 21:57:57,881 [INFO] docsgpt_utils: query_docsgpt | Read the following document carefully and write a concise, well-structured summary in no more than 2
2026-05-06 21:58:16,110 [INFO] docsgpt_utils: answer received (1076 chars)
2026-05-06 21:58:16,112 [INFO] docsgpt_utils: generate_faqs: 2 Qs for 'ui_faq'
2026-05-06 21:58:16,113 [INFO] docsgpt_utils: query_docsgpt | Based on the document below, generate exactly 2 frequently asked questions (FAQs) with detailed, hel
2026-05-06 21:58:17,914 [INFO] docsgpt_utils: answer received (915 chars)
2026-05-06 21:58:17,940 [INFO] absl: Using default tokenizer.
2026-05-06 21:58:18,113 [INFO] docsgpt_utils: evaluate_output: {'rouge1': 0.6117, 'rouge2': 0.402, 'rougeL': 0.4466, 'bleu': 0.2475}


---

## What We Built

| Part | Function used | API call |
|------|---------------|----------|
| Data Collection | `fetch_awesome_ml_readme()`, `load_stackoverflow_sample()`, `load_pile_sample()` | GitHub, HuggingFace Hub |
| Summarisation | `summarize_document()` | `POST /api/answer` |
| FAQ Generation | `generate_faqs()`, `parse_faqs()` | `POST /api/answer` |
| Evaluation | `evaluate_all()` | rouge_score, nltk |
| Multi-Language | `summarize_multilang()`, `generate_faqs_multilang()` | deep-translator + DocsGPT |
| UI | `gr.Blocks()`, `demo.launch()` | Gradio |
